# 16_insumos_adicionales_abm.ipynb
## Insumos faltantes para el ABM: transiciones generales y perfil de entrantes a la informalidad

**Objetivo:**
1. Matriz general de probabilidades de transición para orígenes en Empleo Formal, Desocupación o Inactividad.
2. Distribución conjunta de `cluster_final` en t+1 × cuartil del IVLR en t+1 entre quienes ingresan a Empleo Informal.

**Precisión metodológica:** `cluster_final` e IVLR observados en t+1 describen la nueva inserción informal; **no** se usan para predecir la transición.

**Salidas:** `Salidas/12_insumos_abm/`.

In [1]:
# 1. Configuración y rutas
import os
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path(
    os.getenv(
        "TFM_PROJECT_ROOT",
        r"C:\Users\tomas\OneDrive\Master Granada\TFM",
    )
).expanduser().resolve()

PANEL_PATH = PROJECT_ROOT / "Salidas" / "06_transiciones" / "panel_estados_auditado.rds"
IVLR_PATH = PROJECT_ROOT / "Salidas" / "09_ivlr" / "base_ivlr_preliminar.parquet"
OUT_DIR = PROJECT_ROOT / "Salidas" / "12_insumos_abm"
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert PANEL_PATH.exists(), f"Panel no encontrado: {PANEL_PATH}"
assert IVLR_PATH.exists(), f"IVLR no encontrado: {IVLR_PATH}"

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"OUT_DIR: {OUT_DIR}")

PROJECT_ROOT: C:\Users\tomas\OneDrive\Master Granada\TFM
OUT_DIR: C:\Users\tomas\OneDrive\Master Granada\TFM\Salidas\12_insumos_abm


In [2]:
# 2. Lectura de fuentes (panel .rds y base IVLR)
def leer_rds(ruta: Path) -> pd.DataFrame:
    """Lee un .rds de R. Usa pyreadr si está instalado; si no, Rscript + arrow."""
    ruta = ruta.resolve()
    if not ruta.exists():
        raise FileNotFoundError(f"No existe: {ruta}")

    try:
        import pyreadr
        resultado = pyreadr.read_r(str(ruta))
        df = list(resultado.values())[0]
        if isinstance(df, pd.DataFrame):
            return df
    except Exception as e:
        print(f"Aviso: pyreadr no disponible: {e}")

    # Fallback: Rscript + arrow (requiere R y arrow instalados)
    import subprocess
    import tempfile
    with tempfile.TemporaryDirectory() as td:
        tmp = Path(td) / "panel.parquet"
        codigo = (
            "suppressMessages(library(arrow)); "
            f"x <- readRDS({str(ruta)!r}); "
            f"write_parquet(as.data.frame(x), {str(tmp)!r})"
        )
        proc = subprocess.run(["Rscript", "-e", codigo], capture_output=True, text=True)
        if proc.returncode != 0 or not tmp.exists():
            raise RuntimeError("No se pudo leer el .rds con Rscript.\n" + proc.stderr)
        return pd.read_parquet(tmp)

panel = leer_rds(PANEL_PATH)
ivlr = pd.read_parquet(IVLR_PATH)
print(f"Panel cargado: {panel.shape[0]} filas, {panel.shape[1]} columnas")
print(f"IVLR cargado: {ivlr.shape[0]} filas, {ivlr.shape[1]} columnas")

Panel cargado: 283246 filas, 38 columnas
IVLR cargado: 80300 filas, 12 columnas


In [3]:
# 3. Preparación del panel: id_persona, tipos, ventana, estados y ponderador t0
# Verificar consistencia del panel (ajuste 1)
assert "consistencia" in panel.columns, "El panel no tiene columna consistencia."
assert panel["consistencia"].eq(True).all(), "El panel contiene enlaces no consistentes."

# Normalizar claves y tipos antes de cualquier merge
ESTADOS_VALIDOS = ["Empleo Formal", "Empleo Informal", "Desocupación", "Inactividad"]
ORIGENES_GENERALES = ["Empleo Formal", "Desocupación", "Inactividad"]

estado_t0 = "estado_laboral_clasificado_t0"
estado_t1 = "estado_laboral_clasificado_t1"

panel["id_persona"] = (
    panel["CODUSU"].astype(str) + "_" +
    panel["NRO_HOGAR"].astype(str) + "_" +
    panel["COMPONENTE"].astype(str)
)

# Normalizar id_persona
panel["id_persona"] = panel["id_persona"].astype("string").str.strip()

for col in ["ANO4", "TRIMESTRE", "ANO4_t1", "TRIMESTRE_t1"]:
    if col in panel.columns:
        panel[col] = pd.to_numeric(panel[col], errors="coerce").astype("Int64")

panel[estado_t0] = panel[estado_t0].astype("string").str.strip()
panel[estado_t1] = panel[estado_t1].astype("string").str.strip()

# Ventana de orígenes: 2023T4 - 2025T3 (con máscaras anulables)
mask_ventana = (
    (
        panel["ANO4"].eq(2023)
        & panel["TRIMESTRE"].eq(4)
    )
    | panel["ANO4"].eq(2024)
    | (
        panel["ANO4"].eq(2025)
        & panel["TRIMESTRE"].le(3)
    )
).fillna(False)

mask_estado_t0_valido = panel[estado_t0].isin(ESTADOS_VALIDOS).fillna(False)
mask_estado_t1_valido = panel[estado_t1].isin(ESTADOS_VALIDOS).fillna(False)

panel = panel.loc[
    mask_ventana & mask_estado_t0_valido & mask_estado_t1_valido
].copy()

print(f"Panel en ventana y con estados válidos: {panel.shape[0]} filas")

# Ponderador del trimestre de origen (t0)
if "PONDERA_x" in panel.columns:
    panel["PONDERA_t0"] = pd.to_numeric(panel["PONDERA_x"], errors="coerce")
elif "PONDERA" in panel.columns:
    panel["PONDERA_t0"] = pd.to_numeric(panel["PONDERA"], errors="coerce")
else:
    raise KeyError("No se encontró PONDERA del trimestre de origen en el panel.")

# PONDERA_t1 se toma del panel
assert "PONDERA_t1" in panel.columns, "El panel no contiene PONDERA_t1."
panel["PONDERA_t1"] = pd.to_numeric(panel["PONDERA_t1"], errors="coerce")

Panel en ventana y con estados válidos: 128427 filas


In [4]:
# 4. INSUMO 1: tabla larga de transiciones desde orígenes generales
panel_gen = panel[panel[estado_t0].isin(ORIGENES_GENERALES)].copy()

transiciones = (
    panel_gen
    .groupby([estado_t0, estado_t1], dropna=False)
    .agg(
        n=("id_persona", "size"),
        poblacion_ponderada=("PONDERA_t0", "sum"),
    )
    .reset_index()
    .rename(columns={estado_t0: "estado_origen", estado_t1: "estado_destino"})
)

totales_origen = (
    panel_gen
    .groupby(estado_t0)
    .agg(
        n_origen=("id_persona", "size"),
        poblacion_ponderada_origen=("PONDERA_t0", "sum"),
    )
    .reset_index()
    .rename(columns={estado_t0: "estado_origen"})
)

# Completar combinaciones origen x destino no observadas con cero
idx = pd.MultiIndex.from_product(
    [ORIGENES_GENERALES, ESTADOS_VALIDOS], names=["estado_origen", "estado_destino"]
)
tabla_larga = (
    transiciones
    .set_index(["estado_origen", "estado_destino"])
    .reindex(idx)
    .fillna({"n": 0, "poblacion_ponderada": 0.0})
    .reset_index()
    .merge(totales_origen, on="estado_origen", how="left")
)
tabla_larga["n"] = tabla_larga["n"].astype("Int64")
tabla_larga["probabilidad_no_ponderada"] = tabla_larga["n"] / tabla_larga["n_origen"]
tabla_larga["probabilidad_ponderada"] = (
    tabla_larga["poblacion_ponderada"] / tabla_larga["poblacion_ponderada_origen"]
)
tabla_larga = tabla_larga[[
    "estado_origen", "estado_destino", "n", "poblacion_ponderada",
    "n_origen", "poblacion_ponderada_origen",
    "probabilidad_no_ponderada", "probabilidad_ponderada",
]].sort_values(["estado_origen", "estado_destino"])

print("Tabla larga de transiciones generales:")
print(tabla_larga.to_string(index=False))
tabla_larga.to_csv(OUT_DIR / "transiciones_generales_otros_estados_larga.csv", index=False, encoding="utf-8-sig")

Tabla larga de transiciones generales:
estado_origen  estado_destino     n  poblacion_ponderada  n_origen  poblacion_ponderada_origen  probabilidad_no_ponderada  probabilidad_ponderada
 Desocupación    Desocupación  1203             905570.0      4099                   2916671.0                   0.293486                0.310481
 Desocupación   Empleo Formal   302             251799.0      4099                   2916671.0                   0.073677                0.086331
 Desocupación Empleo Informal  1327             870757.0      4099                   2916671.0                   0.323737                0.298545
 Desocupación     Inactividad  1267             888545.0      4099                   2916671.0                     0.3091                0.304644
Empleo Formal    Desocupación   427             315196.0     35658                  22194865.0                   0.011975                0.014201
Empleo Formal   Empleo Formal 31225           19284389.0     35658                  2

In [5]:
# 5. INSUMO 1: tabla ancha operativa (probabilidades ponderadas) y controles
tabla_operativa = (
    tabla_larga
    .pivot(index="estado_origen", columns="estado_destino", values="probabilidad_ponderada")
    .reset_index()
)
tabla_operativa = tabla_operativa.reindex(columns=["estado_origen"] + ESTADOS_VALIDOS).fillna(0.0)
tabla_operativa.columns.name = None

origen_n = tabla_larga[["estado_origen", "n_origen"]].drop_duplicates()
tabla_operativa = tabla_operativa.merge(origen_n, on="estado_origen", how="left")

tabla_operativa = tabla_operativa.rename(columns={
    "Empleo Formal": "p_empleo_formal",
    "Empleo Informal": "p_empleo_informal",
    "Desocupación": "p_desocupacion",
    "Inactividad": "p_inactividad",
})
cols_prob = ["p_empleo_formal", "p_empleo_informal", "p_desocupacion", "p_inactividad"]
tabla_operativa["suma_probabilidades"] = tabla_operativa[cols_prob].sum(axis=1)
tabla_operativa = tabla_operativa[["estado_origen", "n_origen"] + cols_prob + ["suma_probabilidades"]]

print("Matriz operativa:")
print(tabla_operativa.to_string(index=False))
tabla_operativa.to_csv(OUT_DIR / "transiciones_generales_otros_estados_operativa.csv", index=False, encoding="utf-8-sig")

# Controles del INSUMO 1
assert set(tabla_operativa["estado_origen"]) == {"Empleo Formal", "Desocupación", "Inactividad"}
assert np.allclose(tabla_operativa["suma_probabilidades"], 1.0, atol=1e-8)
print("Controles del INSUMO 1 superados.")

Matriz operativa:
estado_origen  n_origen  p_empleo_formal  p_empleo_informal  p_desocupacion  p_inactividad  suma_probabilidades
 Desocupación      4099         0.086331           0.298545        0.310481       0.304644                  1.0
Empleo Formal     35658         0.868867           0.091947        0.014201       0.024984                  1.0
  Inactividad     60779         0.016574           0.065653        0.027409       0.890363                  1.0
Controles del INSUMO 1 superados.


In [6]:
# 6. INSUMO 2: definición de entrantes a empleo informal
entrantes = panel[
    (panel[estado_t0] != "Empleo Informal") &
    (panel[estado_t0].isin(ORIGENES_GENERALES)) &
    (panel[estado_t1] == "Empleo Informal")
].copy()
entrantes["estado_origen"] = entrantes[estado_t0]
print(f"Entradas a empleo informal (2023T4-2025T3): {len(entrantes)}")
print("Por estado de origen:\n", entrantes["estado_origen"].value_counts())

Entradas a empleo informal (2023T4-2025T3): 8181
Por estado de origen:
 estado_origen
Inactividad      3826
Empleo Formal    3028
Desocupación     1327
Name: count, dtype: int64[pyarrow]


In [7]:
# 7. Cuartiles del IVLR en la base completa (observaciones informales con IVLR observado)
# Normalizar claves y tipos en IVLR
ivlr["id_persona"] = ivlr["id_persona"].astype("string").str.strip()
for columna in ["ANO4", "TRIMESTRE"]:
    ivlr[columna] = pd.to_numeric(ivlr[columna], errors="raise").astype("Int64")

ETIQUETAS_CUARTIL = [
    "Q1 menor vulnerabilidad", "Q2", "Q3", "Q4 mayor vulnerabilidad",
]

# Control de duplicados en la clave de la base IVLR
assert int(ivlr.duplicated(["id_persona", "ANO4", "TRIMESTRE"]).sum()) == 0, \
    "Clave duplicada en la base IVLR."

# Los puntos de corte usan TODAS las observaciones informales con IVLR observado,
# no solamente las personas entrantes.
ivlr = ivlr.copy()
mask_obs = ivlr["ivlr_ampliado"].notna()
cuartiles = (
    ivlr.loc[mask_obs]
    .groupby(["ANO4", "TRIMESTRE"])["ivlr_ampliado"]
    .transform(
        lambda x: pd.qcut(x.rank(method="first"), q=4, labels=ETIQUETAS_CUARTIL)
    )
)
ivlr.loc[mask_obs, "cuartil_ivlr"] = cuartiles
print(f"Base IVLR con cuartil calculado: {int(mask_obs.sum())} observaciones con IVLR observado.")

Base IVLR con cuartil calculado: 64761 observaciones con IVLR observado.


In [8]:
# 8. Renombrar base IVLR para el enlace en t+1 (sin PONDERA)
rename_cols = {
    "ANO4": "ANO4_t1",
    "TRIMESTRE": "TRIMESTRE_t1",
    "cluster_final": "cluster_final_t1",
    "ivlr_ampliado": "ivlr_ampliado_t1",
    "cuartil_ivlr": "cuartil_ivlr_t1",
}
ivlr_t1 = ivlr.rename(columns=rename_cols)
cols_t1 = [
    "id_persona",
    "ANO4_t1",
    "TRIMESTRE_t1",
    "cluster_final_t1",
    "ivlr_ampliado_t1",
    "cuartil_ivlr_t1",
]
ivlr_t1 = ivlr_t1[cols_t1].copy()

# Normalizar claves y tipos en ivlr_t1
ivlr_t1["id_persona"] = ivlr_t1["id_persona"].astype("string").str.strip()
for columna in ["ANO4_t1", "TRIMESTRE_t1"]:
    ivlr_t1[columna] = pd.to_numeric(ivlr_t1[columna], errors="raise").astype("Int64")

assert ivlr_t1.duplicated(["id_persona", "ANO4_t1", "TRIMESTRE_t1"]).sum() == 0

print(f"Base IVLR renombrada para enlace t+1: {ivlr_t1.shape[0]} filas.")

Base IVLR renombrada para enlace t+1: 80300 filas.


In [9]:
# 9. Enlace entrantes x observación de destino (t+1), sin duplicación de filas
# Esta unión usa información de t+1 SOLO para describir el perfil asignado
# después de entrar a la informalidad; no para predecir la transición.
n_antes = len(entrantes)
entrantes = entrantes.merge(
    ivlr_t1,
    on=["id_persona", "ANO4_t1", "TRIMESTRE_t1"],
    how="left",
    validate="many_to_one",
    indicator=True,
)
assert len(entrantes) == n_antes, "El merge de destino duplicó filas."
print("_merge counts:\n", entrantes["_merge"].value_counts())

_merge counts:
 _merge
both          8181
left_only        0
right_only       0
Name: count, dtype: int64


In [10]:
# 10. Ponderador del perfil: se toma del panel longitudinal, no de la base IVLR
assert "PONDERA_t1" in entrantes.columns, "El panel no contiene PONDERA_t1."

entrantes["PONDERA_t1"] = pd.to_numeric(entrantes["PONDERA_t1"], errors="coerce")
entrantes["PONDERA_t0"] = pd.to_numeric(entrantes["PONDERA_t0"], errors="coerce")

mask_ponderador_t1 = entrantes["PONDERA_t1"].notna() & entrantes["PONDERA_t1"].gt(0)
mask_ponderador_t0 = entrantes["PONDERA_t0"].notna() & entrantes["PONDERA_t0"].gt(0)

entrantes["ponderador_usado"] = np.select(
    [
        mask_ponderador_t1,
        ~mask_ponderador_t1 & mask_ponderador_t0,
    ],
    [
        "t1",
        "t0_respaldo",
    ],
    default="sin_ponderador",
)

entrantes["PONDERA_entrante"] = np.where(
    mask_ponderador_t1,
    entrantes["PONDERA_t1"],
    entrantes["PONDERA_t0"],
)

assert (entrantes["ponderador_usado"].ne("sin_ponderador").all())
assert entrantes["PONDERA_entrante"].gt(0).all()

print("Ponderador usado en entrantes:")
print(entrantes["ponderador_usado"].value_counts(dropna=False))

Ponderador usado en entrantes:
ponderador_usado
t1    8181
Name: count, dtype: int64


In [11]:
# 11. Auditoría de entradas y cobertura del enlace en t+1
def auditoria_grupo(g):
    return pd.Series({
        "n_entradas": len(g),
        "n_vinculadas_t1": int((g["_merge"] == "both").sum()),
        "n_ivlr_ampliado_observado": int(g["ivlr_ampliado_t1"].notna().sum()),
        "n_sin_ivlr_ampliado": int(g["ivlr_ampliado_t1"].isna().sum()),
        "n_sin_cluster": int(g["cluster_final_t1"].isna().sum()),
    })

aud_origen = (
    entrantes
    .groupby("estado_origen")
    .apply(auditoria_grupo, include_groups=False)
    .reset_index()
)
aud_origen["cobertura_vinculacion_pct"] = 100 * aud_origen["n_vinculadas_t1"] / aud_origen["n_entradas"]
aud_origen["cobertura_ivlr_pct"] = 100 * aud_origen["n_ivlr_ampliado_observado"] / aud_origen["n_entradas"]

cols_sum = ["n_entradas", "n_vinculadas_t1", "n_ivlr_ampliado_observado", "n_sin_ivlr_ampliado", "n_sin_cluster"]
total = aud_origen[cols_sum].sum()
fila_total = pd.DataFrame([{
    "estado_origen": "TOTAL",
    **total.to_dict(),
    "cobertura_vinculacion_pct": 100 * total["n_vinculadas_t1"] / total["n_entradas"],
    "cobertura_ivlr_pct": 100 * total["n_ivlr_ampliado_observado"] / total["n_entradas"],
}])
auditoria = pd.concat([aud_origen, fila_total], ignore_index=True)

print("Auditoría de entrantes a informalidad:")
print(auditoria.to_string(index=False))
auditoria.to_csv(OUT_DIR / "auditoria_entrantes_informalidad.csv", index=False, encoding="utf-8-sig")

Auditoría de entrantes a informalidad:
estado_origen  n_entradas  n_vinculadas_t1  n_ivlr_ampliado_observado  n_sin_ivlr_ampliado  n_sin_cluster  cobertura_vinculacion_pct  cobertura_ivlr_pct
 Desocupación        1327             1327                       1036                  291              7                      100.0           78.070836
Empleo Formal        3028             3028                       2355                  673             28                      100.0           77.774108
  Inactividad        3826             3826                       3007                  819             28                      100.0           78.593832
        TOTAL        8181             8181                       6398                 1783             63                      100.0           78.205598


In [12]:
# 12. Distribución conjunta cluster x cuartil por estado de origen
# Solo entrantes con cuartil observado; sin imputación. 'Sin cluster' es categoría explícita.
ent_obs = entrantes[entrantes["cuartil_ivlr_t1"].notna()].copy()
ent_obs["cluster_final_t1"] = ent_obs["cluster_final_t1"].astype("string").fillna("Sin cluster")

dist_origen = (
    ent_obs
    .groupby(["estado_origen", "cluster_final_t1", "cuartil_ivlr_t1"], dropna=False)
    .agg(n=("id_persona", "size"), poblacion_ponderada=("PONDERA_entrante", "sum"))
    .reset_index()
)
tot_dist = (
    dist_origen.groupby("estado_origen")[["n", "poblacion_ponderada"]].sum()
    .rename(columns={"n": "n_origen_dist", "poblacion_ponderada": "pob_origen_dist"})
    .reset_index()
)
dist_origen = dist_origen.merge(tot_dist, on="estado_origen", how="left")
dist_origen["probabilidad_no_ponderada"] = dist_origen["n"] / dist_origen["n_origen_dist"]
dist_origen["probabilidad_ponderada"] = dist_origen["poblacion_ponderada"] / dist_origen["pob_origen_dist"]

# Regla de respaldo por origen, no por celda
soporte_por_origen = (
    ent_obs
    .groupby("estado_origen")
    .agg(n_perfiles_observados=("id_persona", "size"))
    .reset_index()
)
soporte_por_origen["usar_nivel"] = np.where(
    soporte_por_origen["n_perfiles_observados"].ge(100),
    "distribucion_por_origen",
    "distribucion_general",
)

# Controles de soporte por origen
assert set(soporte_por_origen["estado_origen"]) == {"Empleo Formal", "Desocupación", "Inactividad"}
assert soporte_por_origen["n_perfiles_observados"].sum() == len(ent_obs)
assert soporte_por_origen["usar_nivel"].isin(
    ["distribucion_por_origen", "distribucion_general"]
).all()

dist_origen = dist_origen.merge(
    soporte_por_origen,
    on="estado_origen",
    how="left",
    validate="many_to_one",
)

dist_origen = dist_origen.sort_values(["estado_origen", "cluster_final_t1", "cuartil_ivlr_t1"])

print("Distribución conjunta por origen:")
print(dist_origen.to_string(index=False))
dist_origen.to_csv(OUT_DIR / "distribucion_entrantes_informalidad_por_origen.csv", index=False, encoding="utf-8-sig")

Distribución conjunta por origen:
estado_origen cluster_final_t1         cuartil_ivlr_t1   n  poblacion_ponderada  n_origen_dist  pob_origen_dist  probabilidad_no_ponderada  probabilidad_ponderada  n_perfiles_observados              usar_nivel
 Desocupación                1 Q1 menor vulnerabilidad  32              15556.0           1036         655974.0                   0.030888                0.023714                   1036 distribucion_por_origen
 Desocupación                1                      Q2  50              24214.0           1036         655974.0                   0.048263                0.036913                   1036 distribucion_por_origen
 Desocupación                1                      Q3  65              35048.0           1036         655974.0                   0.062741                0.053429                   1036 distribucion_por_origen
 Desocupación                1 Q4 mayor vulnerabilidad 204             150469.0           1036         655974.0               

In [13]:
# 13. Distribución general de entrantes (sin distinguir estado de origen)
dist_general = (
    ent_obs
    .groupby(["cluster_final_t1", "cuartil_ivlr_t1"], dropna=False)
    .agg(n=("id_persona", "size"), poblacion_ponderada=("PONDERA_entrante", "sum"))
    .reset_index()
)
dist_general["probabilidad_no_ponderada"] = dist_general["n"] / dist_general["n"].sum()
dist_general["probabilidad_ponderada"] = dist_general["poblacion_ponderada"] / dist_general["poblacion_ponderada"].sum()
dist_general = dist_general.sort_values(["cluster_final_t1", "cuartil_ivlr_t1"])

print("Distribución general de entrantes:")
print(dist_general.to_string(index=False))
dist_general.to_csv(OUT_DIR / "distribucion_general_entrantes_informalidad.csv", index=False, encoding="utf-8-sig")

Distribución general de entrantes:
cluster_final_t1         cuartil_ivlr_t1   n  poblacion_ponderada  probabilidad_no_ponderada  probabilidad_ponderada
               1 Q1 menor vulnerabilidad 336             206115.0                   0.052516                0.054716
               1                      Q2 350             184882.0                   0.054705                0.049079
               1                      Q3 486             280981.0                   0.075961                0.074590
               1 Q4 mayor vulnerabilidad 855             552291.0                   0.133636                0.146612
               2 Q1 menor vulnerabilidad 551             382926.0                   0.086121                0.101652
               2                      Q2 328             194500.0                   0.051266                0.051632
               2                      Q3 316             203350.0                    0.04939                0.053982
               2 Q4 mayor vul

In [14]:
# 14. Reglas de asignación de perfil a nuevos informales (documentación operativa)
# Ajuste 2: regla 3 corregida conceptualmente
reglas = pd.DataFrame({
    "regla": [1, 2, 3],
    "descripcion": [
        (
            "Si el estado de origen tiene al menos 100 perfiles "
            "de entrantes observados, sortear cluster y cuartil "
            "desde su distribución específica."
        ),
        (
            "Si el estado de origen tiene menos de 100 perfiles "
            "observados, usar la distribución general de entrantes."
        ),
        (
            "Si el estado de origen no está disponible o no está "
            "reconocido, usar la distribución general de entrantes."
        ),
    ],
    "umbral": [
        "n_perfiles_observados >= 100",
        "n_perfiles_observados < 100",
        "origen no disponible",
    ],
    "nota": [
        "Probabilidades descriptivas, no causales.",
        "El umbral de 100 es operativo y documentado.",
        (
            "El perfil se asigna después de realizarse "
            "la transición a empleo informal."
        ),
    ],
})
reglas.to_csv(OUT_DIR / "reglas_asignacion_nuevos_informales.csv", index=False, encoding="utf-8-sig")
print(reglas.to_string(index=False))

 regla                                                                                                                             descripcion                       umbral                                                                       nota
     1 Si el estado de origen tiene al menos 100 perfiles de entrantes observados, sortear cluster y cuartil desde su distribución específica. n_perfiles_observados >= 100                                  Probabilidades descriptivas, no causales.
     2                               Si el estado de origen tiene menos de 100 perfiles observados, usar la distribución general de entrantes.  n_perfiles_observados < 100                               El umbral de 100 es operativo y documentado.
     3                              Si el estado de origen no está disponible o no está reconocido, usar la distribución general de entrantes.         origen no disponible El perfil se asigna después de realizarse la transición a empleo informal.


In [15]:
# 15. Controles finales
assert set(tabla_operativa["estado_origen"]) == {"Empleo Formal", "Desocupación", "Inactividad"}
assert np.allclose(tabla_operativa["suma_probabilidades"], 1.0, atol=1e-8)

# La distribución conjunta suma 1 dentro de cada estado de origen
assert np.allclose(dist_origen.groupby("estado_origen")["probabilidad_ponderada"].sum(), 1.0, atol=1e-8)
assert np.allclose(dist_origen.groupby("estado_origen")["probabilidad_no_ponderada"].sum(), 1.0, atol=1e-8)

# La distribución general suma 1
assert np.allclose(dist_general["probabilidad_ponderada"].sum(), 1.0, atol=1e-8)
assert np.allclose(dist_general["probabilidad_no_ponderada"].sum(), 1.0, atol=1e-8)

# Clave IVLR sin duplicados y merge de destino sin duplicación de filas
assert int(ivlr.duplicated(["id_persona", "ANO4", "TRIMESTRE"]).sum()) == 0
assert len(entrantes) == n_antes
print("Controles finales superados.")

Controles finales superados.


In [16]:
# 16. Versiones del entorno
import platform
versiones = pd.DataFrame({
    "componente": ["Python", "pandas", "numpy", "pyarrow", "sistema", "entorno_conda"],
    "version": [
        sys.version.split()[0],
        pd.__version__,
        np.__version__,
        pa.__version__,
        platform.system(),
        os.environ.get("CONDA_DEFAULT_ENV", "desconocido"),
    ],
})
versiones.to_csv(OUT_DIR / "versiones_entorno_insumos_abm.csv", index=False, encoding="utf-8-sig")
print(versiones.to_string(index=False))

   componente version
       Python 3.11.14
       pandas   3.0.5
        numpy   2.3.5
      pyarrow  23.0.1
      sistema Windows
entorno_conda trabajo


In [17]:
# 17. Conclusiones
tot = auditoria[auditoria["estado_origen"] == "TOTAL"].iloc[0]

# Resumen de qué orígenes usan distribución por origen vs general
uso_nivel = (
    soporte_por_origen[
        [
            "estado_origen",
            "n_perfiles_observados",
            "usar_nivel",
        ]
    ]
    .drop_duplicates()
    .sort_values("estado_origen")
    .reset_index(drop=True)
)

texto = (
    "INSUMOS ADICIONALES PARA EL ABM - CONCLUSIONES\n\n"
    f"Entradas totales a empleo informal (2023T4-2025T3): {int(tot['n_entradas'])}\n"
    f"Vinculadas con la base IVLR en t+1: {int(tot['n_vinculadas_t1'])} ({tot['cobertura_vinculacion_pct']:.1f}%)\n"
    f"Con IVLR ampliado observado en t+1: {int(tot['n_ivlr_ampliado_observado'])} ({tot['cobertura_ivlr_pct']:.1f}%)\n"
    f"Sin IVLR ampliado: {int(tot['n_sin_ivlr_ampliado'])}\n"
    f"Sin cluster: {int(tot['n_sin_cluster'])}\n\n"
    "Matriz operativa (probabilidades ponderadas) en:\n"
    "  transiciones_generales_otros_estados_operativa.csv\n\n"
    "Regla de asignación de perfil a nuevos informales:\n"
    "  - Estados de origen con >= 100 perfiles observados usan su propia distribución.\n"
    "  - Estados de origen con < 100 perfiles usan la distribución general de entrantes.\n"
    "  - Aplicación por estado de origen:\n"
)
for _, row in uso_nivel.iterrows():
    texto += f"      {row['estado_origen']}: {row['n_perfiles_observados']} perfiles → {row['usar_nivel']}\n"
texto += (
    "\nNOTAS METODOLÓGICAS:\n"
    "- Probabilidades descriptivas, no causales.\n"
    "- cluster_final e IVLR de t+1 se usan SOLO para "
    "asignar atributos después de la transición,\n"
    "  no para predecir la entrada a la informalidad.\n"
    "- Ponderador preferente del perfil de entrantes: "
    "PONDERA_t1; si falta, PONDERA_t0 documentado.\n"
    "- No se imputaron cluster ni IVLR; "
    "'Sin cluster' se conserva como categoría explícita.\n"
    "- Umbral operativo de 100 observaciones por origen "
    "para utilizar la distribución específica.\n"
)

with open(
    OUT_DIR / "conclusiones_insumos_abm.txt",
    "w",
    encoding="utf-8",
) as archivo:
    archivo.write(texto)

print(texto)

INSUMOS ADICIONALES PARA EL ABM - CONCLUSIONES

Entradas totales a empleo informal (2023T4-2025T3): 8181
Vinculadas con la base IVLR en t+1: 8181 (100.0%)
Con IVLR ampliado observado en t+1: 6398 (78.2%)
Sin IVLR ampliado: 1783
Sin cluster: 63

Matriz operativa (probabilidades ponderadas) en:
  transiciones_generales_otros_estados_operativa.csv

Regla de asignación de perfil a nuevos informales:
  - Estados de origen con >= 100 perfiles observados usan su propia distribución.
  - Estados de origen con < 100 perfiles usan la distribución general de entrantes.
  - Aplicación por estado de origen:
      Desocupación: 1036 perfiles → distribucion_por_origen
      Empleo Formal: 2355 perfiles → distribucion_por_origen
      Inactividad: 3007 perfiles → distribucion_por_origen

NOTAS METODOLÓGICAS:
- Probabilidades descriptivas, no causales.
- cluster_final e IVLR de t+1 se usan SOLO para asignar atributos después de la transición,
  no para predecir la entrada a la informalidad.
- Ponderado

In [18]:
# 18. Resumen solicitado
tot = auditoria[auditoria["estado_origen"] == "TOTAL"].iloc[0]
print("=" * 70)
print("MATRIZ OPERATIVA (3 estados de origen):")
print(tabla_operativa.to_string(index=False))
print("\nENTRADAS A INFORMALIDAD:", int(tot["n_entradas"]))
print(f"COBERTURA DEL ENLACE EN t+1: {tot['cobertura_vinculacion_pct']:.1f}% vinculadas; "
      f"{tot['cobertura_ivlr_pct']:.1f}% con IVLR observado")
print("\nTAMAÑOS POR ESTADO DE ORIGEN:")
print(aud_origen[["estado_origen", "n_entradas"]].to_string(index=False))
print("\nDISTRIBUCIÓN GENERAL cluster x cuartil DE ENTRANTES:")
print(dist_general.to_string(index=False))

# Resumen de uso de nivel por origen
uso_nivel = (
    soporte_por_origen[
        [
            "estado_origen",
            "n_perfiles_observados",
            "usar_nivel",
        ]
    ]
    .drop_duplicates()
    .sort_values("estado_origen")
    .reset_index(drop=True)
)
print("\nREGLA DE ASIGNACIÓN POR ESTADO DE ORIGEN:")
print(uso_nivel.to_string(index=False))
print("\nTodos los controles superados sin errores.")

MATRIZ OPERATIVA (3 estados de origen):
estado_origen  n_origen  p_empleo_formal  p_empleo_informal  p_desocupacion  p_inactividad  suma_probabilidades
 Desocupación      4099         0.086331           0.298545        0.310481       0.304644                  1.0
Empleo Formal     35658         0.868867           0.091947        0.014201       0.024984                  1.0
  Inactividad     60779         0.016574           0.065653        0.027409       0.890363                  1.0

ENTRADAS A INFORMALIDAD: 8181
COBERTURA DEL ENLACE EN t+1: 100.0% vinculadas; 78.2% con IVLR observado

TAMAÑOS POR ESTADO DE ORIGEN:
estado_origen  n_entradas
 Desocupación        1327
Empleo Formal        3028
  Inactividad        3826

DISTRIBUCIÓN GENERAL cluster x cuartil DE ENTRANTES:
cluster_final_t1         cuartil_ivlr_t1   n  poblacion_ponderada  probabilidad_no_ponderada  probabilidad_ponderada
               1 Q1 menor vulnerabilidad 336             206115.0                   0.052516          